A node returns Command(goto=<other_agent_name>, update=<state to hand over>)

In [1]:
from typing import Annotated
from langgraph.types import Command
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.prebuilt import InjectedState

c:\Users\geetikak\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def make_handoff_tool(agent_name: str):
    @tool(f"transfer_to_{agent_name}", description=f"Hand off the task to {agent_name}.")
    def handoff(state: Annotated[dict, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
        tool_message = {
            "role": "tool",
            "content": f"Transferred to {agent_name}",
            "name": f"transfer_to{agent_name}",
            "tool_call_id": tool_call_id,
        }

        return Command(
            goto=agent_name,
            update={"messages": [tool_message]},
            graph=Command.PARENT
        )
    return handoff

In [6]:
pip	install	langgraph-supervisor

  Using cached langgraph_supervisor-0.0.31-py3-none-any.whl.metadata (14 kB)
Using cached langgraph_supervisor-0.0.31-py3-none-any.whl (16 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
pip install langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
load_dotenv()
from langgraph.checkpoint.memory import InMemorySaver

In [24]:
if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY not found in environment variables.")

llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature = 0.2
    )

In [21]:
@tool
def web_search(query: str) -> str:
    """Search web for current query"""
    return f"[search results for: {query}]"

In [22]:
@tool
def run_python(code: str) -> str:
    """Execute Python code and return stdout."""
    return exec_and_capture(code)

In [28]:
research_agent = create_agent(
    llm, tools=[web_search], name="research_agent",
    system_prompt="You	research	facts.	Do	not	write	code.	Report	findings	only.",
)

In [26]:
coder_agent	=	create_agent(
llm,	tools=[run_python],	name="coder_agent",
system_prompt="You	write	and	run	Python.	Do	not	do	open-ended	research.",
)

In [29]:

workflow	=	create_supervisor(
[research_agent,	coder_agent],
model=llm,
prompt=(
"You	supervise	a	research_agent	and	a	coder_agent.	"
"Route	research	questions	to	research_agent,	coding	tasks	to	coder_agent.	"
"Assign	one	agent	at	a	time.	Do	not	do	the	work	yourself."
),
)

In [32]:
app	=	workflow.compile(checkpointer=InMemorySaver())

In [34]:
result	=	app.invoke({"messages":	[{"role":	"user",	"content":	"Find	Python's	creation	year,	then	print	it	doubled."}]})
for	m	in	result["messages"]:
    m.pretty_print()

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [35]:
from	typing	import	Literal
from	langgraph.graph	import	StateGraph,	MessagesState,	START
from langgraph.types import Command

class	TeamState(MessagesState):
    next_agent:	str

def	supervisor_node(state:	TeamState)->Command[Literal["research_agent",	"coder_agent",	"__end__"]]:
        decision	=	llm.invoke([{"role":	"system",	"content":	"Reply	with	exactly	one	word:	research_agent,	coder_agent,	or	done."}, *state["messages"],]).content.strip()
        if	decision	==	"done":
            return	Command(goto="__end__")
            return	Command(goto=decision)
builder	=	StateGraph(TeamState)
builder.add_node("supervisor",	supervisor_node)
builder.add_node("research_agent",	research_agent)
builder.add_node("coder_agent",	coder_agent)

builder.add_edge(START,	"supervisor")
builder.add_edge("research_agent",	"supervisor")
builder.add_edge("coder_agent",	"supervisor")
graph	=	builder.compile()

### Handoff

In [ ]:
# coder agent makes a brand new tool: transfer_to_coder_agent
def make_handoff_tool(agent_name: str):
    @tool(f"transfer_to_{agent_name}", description=f"Hand off the task to {agent_name}.")
    def handoff(state: Annotated[dict, InjectedState],
                tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
        tool_message = {
            "role": "tool",
            "content": f"Transferred to {agent_name}",
            "name": f"transfer_to_{agent_name}",
            "tool_call_id": tool_call_id,
        }
        return Command(
            goto=agent_name,
            update={"messages": [tool_message]},
            graph=Command.PARENT,
        )
    return handoff